# 02 — Preprocessing: Membangun Data Siap Latih

Mengeksekusi keputusan fase EDA (`EDA_REPORT_BAGIAN3.md §6`) menjadi data ter-split
(train/val/test) untuk RM-a/b/c. **Notebook ini mengeksekusi**, EDA hanya memutuskan.

**Alur:** load → drop missing → dedup (NFKC-exact, resolve konflik label→1) → stratified
split 70/15/15 → `clean_text` (placeholder, *setelah* split) → guard anti-leakage → simpan.

| Keputusan | Nilai |
|---|---|
| Kunci dedup | NFKC-exact (keep first) |
| Konflik label | assign 1 |
| Placeholder | `[URL]` / `[MENTION]` / `[NUM]` (angka berdiri sendiri) — special token |
| Split | stratified 70/15/15, seed 42 |
| max_length | 128 (tokenizer `indobert-base-p2`) |
| Target reproduksi EDA | 9.412 baris (L0=7.702 / L1=1.710) |

In [1]:
import sys, json
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path('../src').resolve()))
import preprocessing as pp

RANDOM_SEED = pp.RANDOM_SEED
RAW_PATH = Path('../dataset/raw/data_labeling.csv')
PROC_DIR = Path('../dataset/processed'); PROC_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_DIR = Path('../dataset/splits');   SPLIT_DIR.mkdir(parents=True, exist_ok=True)

print('seed', RANDOM_SEED)
print('special tokens', pp.SPECIAL_TOKENS)

seed 42
special tokens ['[URL]', '[MENTION]', '[NUM]']


## 1. Load & Drop Missing

In [2]:
df = pd.read_csv(RAW_PATH, index_col=0)[['textOriginal', 'label']]
n_raw = len(df)

df = df.dropna(subset=['textOriginal', 'label']).copy()
df['label'] = df['label'].astype(int)
n_after_missing = len(df)

print(f'Data awal              : {n_raw:,}')
print(f'- Missing text/label   : -{n_raw - n_after_missing}')
print(f'Setelah drop missing   : {n_after_missing:,}')

Data awal              : 14,237
- Missing text/label   : -10
Setelah drop missing   : 14,227


## 2. Deduplikasi (NFKC-exact) + Resolusi Konflik Label

Kunci dedup = teks ber-NFKC (menangkap obfuskasi Unicode kelas 1). Grup dengan >1 label diselesaikan ke label 1 **sebelum** dedup.

In [3]:
# hitung grup konflik label (deteksi) sebelum resolusi, untuk pelaporan
_tmp = df.copy()
_tmp['nfkc_key'] = _tmp['textOriginal'].map(pp.normalize_nfkc)
_conf = _tmp.groupby('nfkc_key')['label'].nunique()
n_conf_groups = int((_conf > 1).sum())
n_conf_rows = int(_tmp['nfkc_key'].isin(_conf[_conf > 1].index).sum())

dedup = pp.deduplicate(df)               # tambah nfkc_key, resolve konflik, drop dup
n_dedup = len(dedup)
vc = dedup['label'].value_counts().sort_index()
n0, n1 = int(vc[0]), int(vc[1])

print(f'Grup konflik label (resolusi->1): {n_conf_groups} grup / {n_conf_rows} baris')
print(f'\n=== Before/After dedup ===')
print(f'  Setelah drop missing   : {n_after_missing:,}')
print(f'  - Duplikat (NFKC-exact): -{n_after_missing - n_dedup:,}')
print(f'  Setelah dedup          : {n_dedup:,}')
print(f'\n  Distribusi kelas: L0={n0:,} ({n0/n_dedup*100:.2f}%) | L1={n1:,} ({n1/n_dedup*100:.2f}%)')
print(f'  Rasio           : {n0/n1:.2f}:1')

# reproduksi angka EDA
assert n_dedup == 9412, f'expected 9412, got {n_dedup}'
assert (n0, n1) == (7702, 1710), f'expected (7702,1710), got {(n0, n1)}'
print('\n[OK] Angka EDA (9.412 / 7.702 / 1.710) tereproduksi.')

Grup konflik label (resolusi->1): 3 grup / 32 baris

=== Before/After dedup ===
  Setelah drop missing   : 14,227
  - Duplikat (NFKC-exact): -4,815
  Setelah dedup          : 9,412

  Distribusi kelas: L0=7,702 (81.83%) | L1=1,710 (18.17%)
  Rasio           : 4.50:1

[OK] Angka EDA (9.412 / 7.702 / 1.710) tereproduksi.


## 3. Stratified Split 70/15/15

In [4]:
train_df, val_df, test_df = pp.stratified_split(dedup, seed=RANDOM_SEED)

print(f'{"Split":6s} {"n":>7s} {"L0":>7s} {"L1":>7s} {"L1%":>7s}')
print('-' * 38)
for name, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
    a = int((d.label == 0).sum()); b = int((d.label == 1).sum())
    print(f'{name:6s} {len(d):7,} {a:7,} {b:7,} {b/len(d)*100:6.2f}%')
print('-' * 38)
print(f'{"total":6s} {len(train_df)+len(val_df)+len(test_df):7,}')

# cek stratifikasi: proporsi L1 tiap split ~ 18.17%
base = n1 / n_dedup
for name, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
    assert abs(d.label.mean() - base) < 0.01, f'{name} stratifikasi meleset'
print('\n[OK] Stratifikasi terjaga (L1% tiap split dalam +-1pp).')

Split        n      L0      L1     L1%
--------------------------------------
train    6,588   5,391   1,197  18.17%
val      1,412   1,155     257  18.20%
test     1,412   1,156     256  18.13%
--------------------------------------
total    9,412

[OK] Stratifikasi terjaga (L1% tiap split dalam +-1pp).


## 4. Transformasi Teks (`clean_text`) — Setelah Split

NFKC + placeholder `[URL]`/`[MENTION]`/`[NUM]` (angka berdiri sendiri; brand DORA77 utuh). Deterministik per-baris → aman diterapkan per-subset.

In [5]:
for d in (train_df, val_df, test_df):
    d['text_clean'] = d['textOriginal'].map(pp.clean_text)

# contoh before/after: prioritaskan yang mengandung placeholder / obfuskasi
def show_examples(d, k=8):
    mask = d['text_clean'].str.contains(r'\[URL\]|\[MENTION\]|\[NUM\]', regex=True)
    sample = pd.concat([d[mask].head(k), d[~mask].head(2)])
    for _, r in sample.iterrows():
        print(f'  L{r.label} | RAW  : {str(r.textOriginal)[:80]}')
        print(f'       | CLEAN: {str(r.text_clean)[:80]}')
        print()

print('=== Contoh transformasi (train) ===\n')
show_examples(train_df)

=== Contoh transformasi (train) ===



  L0 | RAW  : Top Up Terbaik Se indonesia Hanya Di https://ourastore.com
       | CLEAN: Top Up Terbaik Se indonesia Hanya Di [URL]

  L1 | RAW  : 11;40 Ini sih harus viral!🤎𝐏𝐑𝐎𝐁𝐄𝐓 𝟖𝟓𝟓🤎
       | CLEAN: [NUM];[NUM] Ini sih harus viral!🤎PROBET [NUM]🤎

  L1 | RAW  : 23;16 Subhanallah,🤎𝐏𝐑𝐎𝐁𝐄𝐓 𝟖𝟓𝟓🤎bikin pengalaman main jadi beda! Ceritanya penuh m
       | CLEAN: [NUM];[NUM] Subhanallah,🤎PROBET [NUM]🤎bikin pengalaman main jadi beda! Ceritanya

  L0 | RAW  : ​@PulcherEtFortisahh. Mm ama junglernya emang ampas
       | CLEAN: [MENTION]. Mm ama junglernya emang ampas

  L0 | RAW  : ​@ahmadravi6730lah emang nya saya menjawab dengan nada ngajak ribut dan tawuran 
       | CLEAN: [MENTION] emang nya saya menjawab dengan nada ngajak ribut dan tawuran gitu wkwk

  L0 | RAW  : ​​​@apaGAterimaloberakti bkent Epic dong aduh bocil bocil tido esok sekolah dek🤣
       | CLEAN: [MENTION] bkent Epic dong aduh bocil bocil tido esok sekolah dek🤣

  L0 | RAW  : 4:05😂😂😂😂
       | CLEAN: [NUM]:[NUM]😂😂😂😂

  L0 

## 5. Guard Anti-Leakage

Dua lapis: (a) `nfkc_key` tidak boleh tumpang tindih antar split (dijamin oleh dedup+partisi); (b) `text_clean` identik lintas-split — bisa muncul karena placeholder mengolapskan template-spam beda-URL/angka. Duplikat lintas-split dibuang dari val/test (train dipertahankan).

In [6]:
# (a) nfkc_key overlap antar split -> harus 0
keys = {n: set(d['nfkc_key']) for n, d in [('train',train_df),('val',val_df),('test',test_df)]}
overlap_key = (keys['train'] & keys['val']) | (keys['train'] & keys['test']) | (keys['val'] & keys['test'])
print(f'Overlap nfkc_key antar split: {len(overlap_key)}')
assert len(overlap_key) == 0

# (b) text_clean identik lintas-split: buang dari val/test bila ada di split lain berprioritas
train_clean = set(train_df['text_clean'])
val_clean = set(val_df['text_clean'])

before = (len(train_df), len(val_df), len(test_df))
val_df  = val_df[~val_df['text_clean'].isin(train_clean)].reset_index(drop=True)
test_df = test_df[~test_df['text_clean'].isin(train_clean | set(val_df['text_clean']))].reset_index(drop=True)
after = (len(train_df), len(val_df), len(test_df))
n_leak_removed = (before[1]-after[1]) + (before[2]-after[2])

print(f'Duplikat text_clean lintas-split dibuang dari val/test: {n_leak_removed}')
print(f'  val : {before[1]:,} -> {after[1]:,}')
print(f'  test: {before[2]:,} -> {after[2]:,}')

# verifikasi akhir: tak ada text_clean yang muncul di >1 split
tc = {n: set(d['text_clean']) for n, d in [('train',train_df),('val',val_df),('test',test_df)]}
assert not ((tc['train'] & tc['val']) | (tc['train'] & tc['test']) | (tc['val'] & tc['test']))
print('\n[OK] Tidak ada text_clean yang bocor lintas-split.')

Overlap nfkc_key antar split: 0
Duplikat text_clean lintas-split dibuang dari val/test: 17
  val : 1,412 -> 1,402
  test: 1,412 -> 1,405

[OK] Tidak ada text_clean yang bocor lintas-split.


## 6. Simpan Output

In [7]:
cols = ['textOriginal', 'text_clean', 'label']
train_df[cols].to_csv(SPLIT_DIR / 'train.csv', index=False)
val_df[cols].to_csv(SPLIT_DIR / 'val.csv', index=False)
test_df[cols].to_csv(SPLIT_DIR / 'test.csv', index=False)

# processed = gabungan berlabel split (traceability)
full = pd.concat([
    train_df[cols].assign(split='train'),
    val_df[cols].assign(split='val'),
    test_df[cols].assign(split='test'),
], ignore_index=True)
full.to_csv(PROC_DIR / 'data_clean.csv', index=False)

print('Tersimpan:')
for p in [SPLIT_DIR/'train.csv', SPLIT_DIR/'val.csv', SPLIT_DIR/'test.csv', PROC_DIR/'data_clean.csv']:
    print(f'  {p}  ({p.stat().st_size/1024:.1f} KB)')

Tersimpan:
  ..\dataset\splits\train.csv  (724.8 KB)
  ..\dataset\splits\val.csv  (155.5 KB)
  ..\dataset\splits\test.csv  (155.1 KB)
  ..\dataset\processed\data_clean.csv  (1086.3 KB)


## 7. Class Weights + Metadata

Class weight `balanced` (sklearn) dihitung dari **train saja** untuk loss berbobot. Semua angka & config disimpan ke `metadata.json`.

In [8]:
class_weights = pp.compute_class_weights(train_df['label'])
print('Class weights (dari train):', class_weights)

def dist(d):
    a = int((d.label == 0).sum()); b = int((d.label == 1).sum())
    return {'n': len(d), 'L0': a, 'L1': b, 'L1_pct': round(b/len(d)*100, 2)}

metadata = {
    'source': str(RAW_PATH),
    'random_seed': RANDOM_SEED,
    'counts': {
        'raw': n_raw,
        'after_missing': n_after_missing,
        'after_dedup': n_dedup,
        'leakage_removed': int(n_leak_removed),
        'final_total': len(full),
    },
    'label_conflict': {'groups': n_conf_groups, 'rows': n_conf_rows, 'policy': 'assign_1'},
    'splits': {'train': dist(train_df), 'val': dist(val_df), 'test': dist(test_df)},
    'class_weights': class_weights,
    'config': {
        'dedup_key': 'nfkc_exact',
        'split_ratios': [0.70, 0.15, 0.15],
        'stratified': True,
        'max_length': 128,
        'model_name': 'indobenchmark/indobert-base-p2',
        'special_tokens': pp.SPECIAL_TOKENS,
        'placeholders': {'url': '[URL]', 'mention': '[MENTION]', 'number': 'standalone \\b\\d+\\b -> [NUM]'},
        'kept_as_is': ['short_comments', 'non_indonesian', 'length_outliers', 'emoji'],
        'lowercase_manual': False,
    },
    'notes': 'Index FAISS RM-c HANYA dari train set. Model wajib resize_token_embeddings(len(tokenizer)).',
}
with open(PROC_DIR / 'metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
print('\nmetadata.json tersimpan.')
print(json.dumps(metadata['counts'], indent=2))

Class weights (dari train): {0: 0.6110183639398998, 1: 2.7518796992481205}

metadata.json tersimpan.
{
  "raw": 14237,
  "after_missing": 14227,
  "after_dedup": 9412,
  "leakage_removed": 17,
  "final_total": 9395
}


## 8. Smoke Test Tokenisasi

Memverifikasi `src/dataset.py`: placeholder jadi 1 token, panjang token `text_clean` aman untuk max_length=128, dan batch DataLoader berbentuk benar.

In [9]:
from dataset import load_tokenizer, GamblingCommentDataset

tokenizer = load_tokenizer()
print('vocab size (base+special):', len(tokenizer))
for t in pp.SPECIAL_TOKENS:
    ids = tokenizer.encode(t, add_special_tokens=False)
    print(f'  {t} -> {ids} ({len(ids)} token)')
    assert len(ids) == 1

# panjang token pada text_clean train (konfirmasi ulang max_length=128 aman)
tl = train_df['text_clean'].map(lambda s: len(tokenizer.encode(s, add_special_tokens=True, truncation=False)))
print('\nPanjang token text_clean (train):')
for p in [50, 95, 99, 100]:
    print(f'  P{p:3d}: {np.percentile(tl, p):.0f}')
print(f'  > 128 token: {(tl > 128).mean()*100:.2f}%')

# Dataset + DataLoader
from torch.utils.data import DataLoader
ds = GamblingCommentDataset(train_df['text_clean'], train_df['label'], tokenizer=tokenizer)
batch = next(iter(DataLoader(ds, batch_size=4)))
print('\nbatch input_ids:', tuple(batch['input_ids'].shape), '| labels:', batch['labels'].tolist())
assert tuple(batch['input_ids'].shape) == (4, 128)
print('\n[OK] Tokenisasi & Dataset siap untuk fase modeling.')

C:\Penelitian\IndoBERT-with-RAC\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


vocab size (base+special): 30524
  [URL] -> [30521] (1 token)
  [MENTION] -> [30522] (1 token)
  [NUM] -> [30523] (1 token)



Panjang token text_clean (train):
  P 50: 10
  P 95: 32
  P 99: 58
  P100: 254
  > 128 token: 0.08%

batch input_ids: (4, 128) | labels: [0, 0, 0, 0]

[OK] Tokenisasi & Dataset siap untuk fase modeling.


## Ringkasan

Data siap latih tersimpan di `dataset/splits/` (+ `dataset/processed/data_clean.csv`, `metadata.json`).

- **9.412** baris final (dikurangi guard leakage) — reproduksi keputusan EDA.
- Stratified 70/15/15, seed 42; distribusi kelas terjaga (~18% L1).
- `text_clean` = NFKC + placeholder; **class weight** tersimpan untuk loss berbobot.
- **Langkah berikut** (`03_rma_finetune.ipynb`): impor `dataset.py`, muat model p2, **wajib**
  `model.resize_token_embeddings(len(tokenizer))`. Index FAISS RM-c hanya dari **train**.